# Homework 5: Deploying Machine Learning Models

Worked solution for the 2026 deployment homework. We use the **frozen course artifact** and its pinned environment so inference matches the assignment. The notebook checks the model checksum, loads it for a local prediction, calls the FastAPI service, reads the container configuration, and runs the same API inside Docker.

Run from this folder with the `Python (ML Zoomcamp HW5)` kernel. The Question 6 cell builds and briefly starts a Docker container, then stops it when complete.

## Setup and artifact integrity

`pipeline.bin` is a serialized scikit-learn pipeline. Its checksum tells us we have the exact model artifact expected by this release. Missing numeric values are replaced by the training-set medians stored in `feature_defaults.json`; missing categories become `NA`.

In [1]:
from pathlib import Path
import hashlib
import json
import re
import subprocess
import sys
import time
import tomllib
import urllib.request

ROOT = Path.cwd()
with (ROOT / 'model_metadata.json').open() as f:
    metadata = json.load(f)
actual_sha256 = hashlib.sha256((ROOT / 'pipeline.bin').read_bytes()).hexdigest()
print('Artifact SHA-256:', actual_sha256)
print('Matches release metadata:', actual_sha256 == metadata['model_sha256'])
print('Recorded numeric defaults:', metadata['numeric_defaults'])

Artifact SHA-256: 1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1
Matches release metadata: True
Recorded numeric defaults: {'number_of_courses_viewed': 2.0, 'annual_income': 55051.0, 'interaction_count': 4.0, 'lead_score': 0.49}


## Question 1 — environment check

`uv` is the Python project and environment manager used by this homework. The assignment asks us to record its local version; it is a setup note, not a model answer.

In [2]:
uv_version = subprocess.check_output(['uv', '--version'], text=True).strip()
print(uv_version)

uv 0.11.26 (x86_64-unknown-linux-gnu)


## Question 2 — locked dependency

The project declares exact dependencies in `pyproject.toml`, and `uv.lock` records the resolved environment. The frozen reference uses Scikit-Learn **1.7.2**. Keep the lockfile as provided and use `uv sync --locked` so the model is loaded with its reference library version.

In [3]:
with (ROOT / 'pyproject.toml').open('rb') as f:
    project = tomllib.load(f)
scikit_learn_requirement = next(dep for dep in project['project']['dependencies'] if dep.startswith('scikit-learn=='))
locked_package = re.search(r'(?m)^name = "scikit-learn"\nversion = "([^"]+)"', (ROOT / 'uv.lock').read_text())
locked_version = locked_package.group(1)
print('Declared:', scikit_learn_requirement)
print('Locked:', locked_version)

Declared: scikit-learn==1.7.2
Locked: 1.7.2


## Question 3 — load the model

We deserialize the frozen pipeline with `pickle`, normalize the lead using the same missing-value rules as the API, and ask for the probability of class 1 (`converted`). The score is **0.532994…**, which rounds to **0.533**. The returned boolean is a separate decision made at a 0.5 cutoff.

In [4]:
import pickle
from model import normalize_record

lead_q3 = {
    'lead_source': 'paid_ads',
    'industry': 'technology',
    'employment_status': 'employed',
    'location': 'north_america',
    'number_of_courses_viewed': 2,
    'annual_income': 79276.0,
    'interaction_count': 4,
    'lead_score': 0.41,
}
with (ROOT / 'pipeline.bin').open('rb') as f:
    pipeline = pickle.load(f)
record = normalize_record(lead_q3)
probability_q3 = float(pipeline.predict_proba([record])[0, 1])
print(f'Conversion probability: {probability_q3:.6f}')
print(f'Rounded answer: {probability_q3:.3f}')
print(f'Conversion at 0.5 cutoff: {probability_q3 >= 0.5}')

Conversion probability: 0.532994
Rounded answer: 0.533
Conversion at 0.5 cutoff: True


## Question 4 — serve the model

The FastAPI app exposes `/health` for a readiness and checksum check and `/predict` for inference. This cell starts the **real Uvicorn service** as a subprocess, sends the homework's second lead over HTTP, and then shuts the temporary service down. The response probability rounds to **0.770**.

In [5]:
lead_q4 = {
    'lead_source': 'organic_search',
    'industry': 'technology',
    'employment_status': 'employed',
    'location': 'europe',
    'number_of_courses_viewed': 4,
    'annual_income': 80304.0,
    'interaction_count': 7,
    'lead_score': 0.74,
}
base_url = 'http://127.0.0.1:9696'
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'predict:app', '--host', '127.0.0.1', '--port', '9696'],
    cwd=ROOT,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
try:
    for _ in range(40):
        if server.poll() is not None:
            raise RuntimeError('Uvicorn exited before becoming ready; check whether port 9696 is already in use.')
        try:
            with urllib.request.urlopen(base_url + '/health', timeout=1) as response:
                health_q4 = json.load(response)
            break
        except Exception:
            time.sleep(0.25)
    else:
        raise TimeoutError('API did not become ready on port 9696')

    request = urllib.request.Request(
        base_url + '/predict',
        data=json.dumps(lead_q4).encode(),
        headers={'Content-Type': 'application/json'},
        method='POST',
    )
    with urllib.request.urlopen(request, timeout=5) as response:
        result_q4 = json.load(response)
    print('Health:', health_q4)
    print('Prediction:', result_q4)
    print(f"Rounded answer: {result_q4['conversion_probability']:.3f}")
finally:
    server.terminate()
    server.wait(timeout=10)

Health: {'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}
Prediction: {'conversion_probability': 0.769799, 'conversion': True}
Rounded answer: 0.770


## Question 5 — inspect the container configuration

The Dockerfile declares the Python base image directly in its `FROM` line. The answer is **`python:3.11.15-slim-bookworm`** (the file also pins the image by digest).

In [6]:
dockerfile = (ROOT / 'Dockerfile').read_text()
base_image = re.search(r'(?m)^FROM (python:[^@\s]+)', dockerfile).group(1)
print('Docker base image:', base_image)

Docker base image: python:3.11.15-slim-bookworm


## Question 6 — run the container

The container should produce the same prediction as the local API because it packages the same artifact and locked dependencies. This cell builds the homework image, starts it on port 9696, checks `/health` and its model checksum, calls the provided smoke client, and stops the temporary container. The probability rounds to **0.770**.

In [7]:
image_tag = 'zoomcamp-model:2026-hw5'
container_name = 'mlzoomcamp-hw5-notebook-check'
subprocess.run(['docker', 'build', '-t', image_tag, '.'], cwd=ROOT, check=True, capture_output=True, text=True)
container_id = subprocess.check_output(
    ['docker', 'run', '--rm', '-d', '--name', container_name, '-p', '9696:9696', image_tag],
    cwd=ROOT, text=True,
).strip()
try:
    for _ in range(40):
        try:
            with urllib.request.urlopen(base_url + '/health', timeout=1) as response:
                health_q6 = json.load(response)
            break
        except Exception:
            time.sleep(0.25)
    else:
        raise TimeoutError('Container API did not become ready on port 9696')

    q6_output = subprocess.check_output([sys.executable, 'q6_test.py'], cwd=ROOT, text=True).strip()
    import ast
    result_q6 = ast.literal_eval(q6_output)
    assert health_q6['status'] == 'ok'
    assert health_q6['model_sha256'] == metadata['model_sha256']
    print('Health:', health_q6)
    print('Prediction:', result_q6)
    print(f"Rounded answer: {result_q6['conversion_probability']:.3f}")
finally:
    subprocess.run(['docker', 'stop', container_id], check=False, capture_output=True, text=True)

Health: {'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}
Prediction: {'conversion_probability': 0.769799, 'conversion': True}
Rounded answer: 0.770


## Answer summary

| Question | Answer |
|---|---|
| 1 | `uv 0.11.26` on this machine (setup check only) |
| 2 | Scikit-Learn **1.7.2** |
| 3 | Conversion probability **0.533** |
| 4 | API conversion probability **0.770** |
| 5 | `python:3.11.15-slim-bookworm` |
| 6 | Container conversion probability **0.770** |

The boolean `conversion` is `true` for these leads because their probability is at least 0.5. A probability is the model's score; the boolean is the API's thresholded decision.